In [ ]:
# See what code was run in each backend
report = pr.compare("ols", "y ~ x1 + x2", df, vcov="HC1", backend=["pyfixest", "statsmodels"])
for name, result in report.backends.items():
    print(f"--- {name} ---")
    print(result.code)
    print()

### Inspecting Backend Code

Each backend result includes the equivalent code that was run:

In [ ]:
# Panel FE — compare against linearmodels and R
report = pr.compare(
    "panel_fe", "y ~ x1 + x2", df,
    entity="firm_id", time="year_id",
    backend=["linearmodels", "r"],
    rtol=5e-2,
)
print(report.summary())

### Panel FE comparison

In [ ]:
# Logit — compare against statsmodels
report = pr.compare(
    "logit", "y_binary ~ x1 + x2", df,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-3,
)
print(report.summary())

In [ ]:
# Probit — compare against pyfixest and statsmodels
report = pr.compare(
    "probit", "y_binary ~ x1 + x2", df,
    backend=["pyfixest", "statsmodels"],
    rtol=1e-3,
)
print(report.summary())

### Probit / Logit comparison

In [ ]:
report = pr.compare(
    "ols", "y ~ x1 + x2 | firm_id", df,
    cluster=["firm_id"],
    backend=["pyfixest", "r"],
    rtol=1e-4,
)
print(report.summary())

### OLS with Fixed Effects + Clustering

In [ ]:
# Default: runs all available backends (pyfixest, statsmodels, R, Stata, linearmodels)
# Skips any that aren't installed
report = pr.compare("ols", "y ~ x1 + x2", df, vcov="HC1")
print(report.summary())

---

## 2. `compare()` — Cross-Package Parity

The new `compare()` function runs the same regression in multiple packages and shows a side-by-side comparison.

### OLS comparison across all available backends

In [ ]:
print(pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"]).as_latex())

### LaTeX Export

Use `.as_latex()` to get a LaTeX string for papers:

In [ ]:
# Transposed layout (models as rows)
pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE"], transpose=True)

In [ ]:
# Show both t-stats and SEs, with standard errors in brackets
pr.regtable(r1, r2, stat=("t", "se"), rename={"_cons": "Constant"})

### Display Options

All the existing display options still work: `stat`, `brackets`, `wide`, `transpose`, `rename`, `model_type`:

In [ ]:
# Add a title, customize styling
table = pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"])
table.tab_header(title="Table 1: OLS Results", subtitle="Dependent variable: y")

### GT Customization

Since `regtable()` returns a GT object, you can chain GT methods for further customization:

In [ ]:
# Fit several models
r1 = pr.ols("y ~ x1 + x2", data=df)
r2 = pr.ols("y ~ x1 + x2", data=df, vcov="HC1")
r3 = pr.ols("y ~ x1 + x2 | firm_id", data=df, cluster=["firm_id"])

# Side-by-side table — renders as a styled GT table in Jupyter
pr.regtable(r1, r2, r3, labels=["OLS", "Robust", "FE+Cluster"])

## 1. `regtable()` — Great Tables Integration

`regtable()` now returns a `great_tables.GT` object instead of a string. It renders natively in Jupyter:

In [ ]:
import numpy as np
import polars as pl
import polars_reg as pr

# Generate sample data
rng = np.random.default_rng(42)
n = 1000

firm_id = np.repeat(np.arange(50), 20)
year_id = np.tile(np.arange(2000, 2020), 50)
fe = rng.standard_normal(50)[firm_id] * 0.5

x1 = rng.standard_normal(n)
x2 = rng.standard_normal(n)
z1 = rng.standard_normal(n)
u = rng.standard_normal(n) * 0.5
x_endog = 0.5 * z1 + 0.3 * u
y = 1.0 + 2.0 * x1 - 0.5 * x2 + 0.8 * x_endog + fe + u

prob = 1.0 / (1.0 + np.exp(-(0.5 + 1.0 * x1 - 0.3 * x2)))
y_binary = (rng.uniform(size=n) < prob).astype(float)

df = pl.DataFrame({
    "y": y, "x1": x1, "x2": x2, "x_endog": x_endog, "z1": z1,
    "y_binary": y_binary, "firm_id": firm_id, "year_id": year_id,
})
print(f"Dataset: {df.shape[0]} rows, {df.shape[1]} columns")

# polars_reg v0.2.0 — New Features Showcase

This notebook demonstrates the key new features in v0.2.0:

1. **`regtable()` via Great Tables** — returns a GT object with native Jupyter rendering, LaTeX/HTML export
2. **`compare()` — unified cross-package comparison** — run the same regression in pyfixest, statsmodels, linearmodels, R, and Stata

---

## Setup